In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Load and Inspect the Dataset

In [5]:
path = "job_salary_estimator/jobs_himalayas_1500.csv"
df_raw = pd.read_csv(path)

df_raw.head(10)

,source,guid,title,company,employment_type,seniority,category,parent_categories,location_restrictions,timezone_restrictions,salary_currency,salary_min,salary_max,pub_date,expiry_date,application_link
0,himalayas,https://himalayas.app/companies/crowdstrike/jo...,"Sr. Business Analyst, Go-to-Market",Crowdstrike,Full Time,Senior,NaN,NaN,United States,-10|-9|-8|-7|-6|-5|14,USD,125000,180000,1770307225,1775491224,https://himalayas.app/companies/crowdstrike/jo...
1,himalayas,https://himalayas.app/companies/nox-medical/jo...,"Director, Product Marketing",Nox Medical,Full Time,Director,NaN,NaN,United States,-10|-9|-8|-7|-6|-5|14,USD,140000,150000,1770307203,1775491203,https://himalayas.app/companies/nox-medical/jo...
2,himalayas,https://himalayas.app/companies/state-of-michi...,Barry County Assistance Payments Worker 8-11 (13),State of Michigan,Full Time,Entry-level,NaN,NaN,United States,-10|-9|-8|-7|-6|-5|14,USD,48526,70657,1770307182,1775195389,https://himalayas.app/companies/state-of-michi...
3,himalayas,https://himalayas.app/companies/ge-healthcare/...,Technical Support Engineer II - HL7,GE HealthCare,Full Time,Mid-level,NaN,Developer,United States,-10|-9|-8|-7|-6|-5|14,USD,108000,162000,1770307008,1775491008,https://himalayas.app/companies/ge-healthcare/...
4,himalayas,https://himalayas.app/companies/shi-internatio...,PubSec Project Admin,SHI International Corp.,Part Time,Entry-level,NaN,NaN,United States,-10|-9|-8|-7|-6|-5|14,USD,41600,41600,1770307008,1775491008,https://himalayas.app/companies/shi-internatio...
5,himalayas,https://himalayas.app/companies/wonder/jobs/ac...,Account Executive,Wonder,Full Time,Mid-level,NaN,Sales,United States,-10|-9|-8|-7|-6|-5|14,USD,50000,100000,1770307008,1775491008,https://himalayas.app/companies/wonder/jobs/ac...
6,himalayas,https://himalayas.app/companies/teecom/jobs/as...,Associate - Audiovisual (UI Programming),TEECOM,Full Time,Entry-level,NaN,NaN,United States,-10|-9|-8|-7|-6|-5|14,USD,110000,140000,1770306821,1775490820,https://himalayas.app/companies/teecom/jobs/as...
7,himalayas,https://himalayas.app/companies/mindrift/jobs/...,Freelance Electrical Engineer with Python Expe...,Mindrift,Part Time,Mid-level,NaN,Hardware Engineer,South Africa,2,USD,41600,41600,1770306809,1775490808,https://himalayas.app/companies/mindrift/jobs/...
8,himalayas,https://himalayas.app/companies/mm-coaching-li...,Video Editor,MM Coaching Limited,Full Time,Entry-level|Mid-level,NaN,Content Creator,Brazil,-5|-4|-3|-2,USD,14400,20400,1770306798,1775490797,https://himalayas.app/companies/mm-coaching-li...
9,himalayas,https://himalayas.app/companies/intellishift/j...,Account Development Representative,IntelliShift,Full Time,Entry-level,NaN,Sales,United States,-10|-9|-8|-7|-6|-5|14,USD,60000,60000,1770306753,1775490752,https://himalayas.app/companies/intellishift/j...


# Data Cleaning

### Data Profiling: Missingness Check

In [ ]:
# decide which columns are worth keeping
missing = df_raw.isna().mean().sort_values(ascending=False)
missing.head(15)

category                 1.0000
parent_categories        0.5998
location_restrictions    0.0096
source                   0.0000
guid                     0.0000
title                    0.0000
company                  0.0000
employment_type          0.0000
seniority                0.0000
timezone_restrictions    0.0000
salary_currency          0.0000
salary_min               0.0000
salary_max               0.0000
pub_date                 0.0000
expiry_date              0.0000
dtype: float64

### Column Selection

In [ ]:
# We keep guid + application_link temp for dedup
keep_cols = [
    "guid",
    "application_link",
    "title",
    "company",
    "employment_type",
    "seniority",
    "parent_categories",
    "location_restrictions",
    "salary_currency",
    "salary_min",
    "salary_max",
]

df_clean = df_raw[[c for c in keep_cols if c in df_raw.columns]].copy()

df_clean = df_clean.drop(columns=["category"], errors="ignore")


print("After column selection:", df_clean.shape)
df_clean.head(3)

After column selection: (5000, 11)


,guid,application_link,title,company,employment_type,seniority,parent_categories,location_restrictions,salary_currency,salary_min,salary_max
0,https://himalayas.app/companies/crowdstrike/jo...,https://himalayas.app/companies/crowdstrike/jo...,"Sr. Business Analyst, Go-to-Market",Crowdstrike,Full Time,Senior,NaN,United States,USD,125000,180000
1,https://himalayas.app/companies/nox-medical/jo...,https://himalayas.app/companies/nox-medical/jo...,"Director, Product Marketing",Nox Medical,Full Time,Director,NaN,United States,USD,140000,150000
2,https://himalayas.app/companies/state-of-michi...,https://himalayas.app/companies/state-of-michi...,Barry County Assistance Payments Worker 8-11 (13),State of Michigan,Full Time,Entry-level,NaN,United States,USD,48526,70657


### Standardize Types + Normalize Strings

In [ ]:
# Standardize text columns to string dtype
text_cols = [
    "guid", "application_link", "title", "company",
    "employment_type", "seniority", "parent_categories",
    "location_restrictions", "salary_currency"
]
for c in text_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].astype("string")

# Convert salary columns to numeric
df_clean["salary_min"] = pd.to_numeric(df_clean["salary_min"], errors="coerce")
df_clean["salary_max"] = pd.to_numeric(df_clean["salary_max"], errors="coerce")

# Normalize currency
df_clean["salary_currency"] = df_clean["salary_currency"].str.upper()

df_clean.isna().mean().sort_values(ascending=False).head(10)

parent_categories        0.5998
location_restrictions    0.0096
guid                     0.0000
application_link         0.0000
title                    0.0000
company                  0.0000
employment_type          0.0000
seniority                0.0000
salary_currency          0.0000
salary_min               0.0000
dtype: float64

### Deduplication

In [ ]:
# Deduplicate by guid first, then application_link
if "guid" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["guid"]).copy()

if "application_link" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["application_link"]).copy()

print("After dedup:", df_clean.shape)

After dedup: (5000, 11)


### Create Target Salary

In [ ]:
# Build target salary:
# - if both min & max exist -> midpoint
# - else use whichever exists
df_clean["salary_target"] = np.where(
    df_clean["salary_min"].notna() & df_clean["salary_max"].notna(),
    (df_clean["salary_min"] + df_clean["salary_max"]) / 2.0,
    np.where(df_clean["salary_min"].notna(), df_clean["salary_min"], df_clean["salary_max"])
)

df_clean = df_clean[df_clean["salary_target"].notna()].copy()

print("After target creation:", df_clean.shape)
df_clean["salary_target"].describe()

After target creation: (5000, 12)


count    5.000000e+03
mean     2.561892e+05
std      5.137834e+06
min      1.138000e+03
25%      4.888000e+04
50%      9.411500e+04
75%      1.450000e+05
max      2.530944e+08
Name: salary_target, dtype: float64

### Filter Currency + Remove Outliers

In [ ]:
# Keep only USD
df_clean = df_clean[df_clean["salary_currency"].isin(["USD", "$"])].copy()

# Remove extreme outliers
LOW, HIGH = 15000, 600000
df_clean = df_clean[(df_clean["salary_target"] >= LOW) & (df_clean["salary_target"] <= HIGH)].copy()

print("After currency+outlier filters:", df_clean.shape)
df_clean["salary_target"].describe()

After currency+outlier filters: (4704, 12)


count      4704.000000
mean     109168.796237
std       72274.668071
min       15000.000000
25%       52500.000000
50%       95680.000000
75%      145500.000000
max      600000.000000
Name: salary_target, dtype: float64

### Final Feature Prep

In [ ]:
# Fill missing text fields
for c in ["title", "company", "employment_type", "seniority", "parent_categories", "location_restrictions"]:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna("")

# Build a focused text field for TF-IDF
df_clean["text_all"] = df_clean["title"] + " | " + df_clean["parent_categories"]

### Final Modeling Dataset

In [ ]:
# Final modeling dataset
df_model_v2 = df_clean[[
    "text_all",
    "company",
    "location_restrictions",
    "employment_type",
    "seniority",
    "salary_target"
]].copy()

print("df_model_v2 shape:", df_model_v2.shape)

pd.set_option("display.max_colwidth", None)

df_model_v2.to_csv("jobs_clean_for_model_v2.csv", index=False)

df_model_v2.head(10)

df_model_v2 shape: (4704, 6)


,text_all,company,location_restrictions,employment_type,seniority,salary_target
0,"Sr. Business Analyst, Go-to-Market |",Crowdstrike,United States,Full Time,Senior,152500.0
1,"Director, Product Marketing |",Nox Medical,United States,Full Time,Director,145000.0
2,Barry County Assistance Payments Worker 8-11 (13) |,State of Michigan,United States,Full Time,Entry-level,59591.5
3,Technical Support Engineer II - HL7 | Developer,GE HealthCare,United States,Full Time,Mid-level,135000.0
4,PubSec Project Admin |,SHI International Corp.,United States,Part Time,Entry-level,41600.0
5,Account Executive | Sales,Wonder,United States,Full Time,Mid-level,75000.0
6,Associate - Audiovisual (UI Programming) |,TEECOM,United States,Full Time,Entry-level,125000.0
7,Freelance Electrical Engineer with Python Experience - AI Trainer | Hardware Engineer,Mindrift,South Africa,Part Time,Mid-level,41600.0
8,Video Editor | Content Creator,MM Coaching Limited,Brazil,Full Time,Entry-level|Mid-level,17400.0
9,Account Development Representative | Sales,IntelliShift,United States,Full Time,Entry-level,60000.0
